# GDELT Extraction — full dataset (all countries, all event types)

Streamlined, self-contained version of `GDELT_Extraction.ipynb` — only the cells needed to
produce one clean events file. Two filters, both exposed in the config cell below instead of
hard-coded:

- `COUNTRIES` — list of ISO3 codes to restrict to, or `None` for every country.
- `EVENT_ROOT_CODES` — list of CAMEO root codes (e.g. `['14','17','18','19','20']`) to restrict
  to, or `None` for every event type.

Both default to `None` (unrestricted) below. Only `MIN_MENTIONS`/`MIN_SOURCES` still apply, as a
light noise floor — same thresholds the original pull used.

**Why unrestricted event types is the recommended default:** the benchmark notebooks that build
`gdelt_bilateral_by_pair_year.parquet` and the country-year GDELT feature files already re-apply
`CRIT = ['14','17','18','19','20']` themselves after loading the raw events parquet. Pulling
everything here doesn't change any existing result — it just moves the one open decision (which
event categories count as "risk") entirely into that downstream step, where it already lives,
instead of also baking it into the raw extraction. You can revisit that choice later without
re-downloading anything.

In [2]:
!pip install pandas tqdm pyarrow requests -q

Error processing line 1 of /home/hefouzinho/miniconda3/envs/mathematical_implementations/lib/python3.8/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "/home/hefouzinho/miniconda3/envs/mathematical_implementations/lib/python3.8/site.py", line 177, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored


In [3]:
import os, io, zipfile, time
import pandas as pd
import requests
from datetime import datetime, timedelta
from tqdm import tqdm

## Config — the only cell you should need to edit

In [4]:
START_YEAR, END_YEAR = 2017, 2023

COUNTRIES = None          # e.g. ['TWN','CHN','USA'] to restrict -- None = every country
EVENT_ROOT_CODES = None   # e.g. ['14','17','18','19','20'] to restrict -- None = every event type

MIN_MENTIONS = 5
MIN_SOURCES = 2

CHECKPOINT_DIR = "gdelt_chunks_full"
OUTPUT_PATH = "gdelt_events_2017_2023.parquet"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [5]:
# GDELT 1.0 daily files = 58 columns (no ADM2 fields; DATEADDED is YYYYMMDD)
GDELT_COLUMNS_V1 = [
    "GLOBALEVENTID","SQLDATE","MonthYear","Year","FractionDate",
    "Actor1Code","Actor1Name","Actor1CountryCode","Actor1KnownGroupCode",
    "Actor1EthnicCode","Actor1Religion1Code","Actor1Religion2Code",
    "Actor1Type1Code","Actor1Type2Code","Actor1Type3Code",
    "Actor2Code","Actor2Name","Actor2CountryCode","Actor2KnownGroupCode",
    "Actor2EthnicCode","Actor2Religion1Code","Actor2Religion2Code",
    "Actor2Type1Code","Actor2Type2Code","Actor2Type3Code",
    "IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass",
    "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone",
    "Actor1Geo_Type","Actor1Geo_FullName","Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code","Actor1Geo_Lat","Actor1Geo_Long","Actor1Geo_FeatureID",
    "Actor2Geo_Type","Actor2Geo_FullName","Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code","Actor2Geo_Lat","Actor2Geo_Long","Actor2Geo_FeatureID",
    "ActionGeo_Type","ActionGeo_FullName","ActionGeo_CountryCode",
    "ActionGeo_ADM1Code","ActionGeo_Lat","ActionGeo_Long","ActionGeo_FeatureID",
    "DATEADDED","SOURCEURL",
]

KEEP_COLS = [
    "GLOBALEVENTID","SQLDATE","DATEADDED","Actor1Name","Actor1CountryCode",
    "Actor2Name","Actor2CountryCode","ActionGeo_CountryCode","ActionGeo_FullName",
    "EventCode","EventBaseCode","EventRootCode",
    "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone","SOURCEURL",
]

In [6]:
def generate_gdelt_dates(start_date, end_date):
    cur = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    days = []
    while cur <= end:
        days.append(cur.strftime("%Y%m%d")); cur += timedelta(days=1)
    return days

def month_ranges(y0, y1):
    out, cur, end = [], datetime(y0,1,1), datetime(y1,12,31)
    while cur <= end:
        nxt = datetime(cur.year + (cur.month==12), (cur.month % 12)+1, 1)
        out.append((cur.strftime("%Y-%m-%d"), (nxt-timedelta(days=1)).strftime("%Y-%m-%d")))
        cur = nxt
    return out

In [7]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 (research data pull)"})

def download_gdelt_file_v1(date_str, retries=4, pause=0.4):
    """One GDELT 1.0 daily file, politely -- avoids the rate-limit block."""
    url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=60)
            if r.status_code == 200:
                z = zipfile.ZipFile(io.BytesIO(r.content))
                raw = pd.read_csv(z.open(z.namelist()[0]), sep="\t",
                                  header=None, dtype=str, low_memory=False)
                if raw.shape[1] != len(GDELT_COLUMNS_V1):
                    print(f"  ⚠ {date_str}: {raw.shape[1]} cols (expected {len(GDELT_COLUMNS_V1)})")
                    return None
                raw.columns = GDELT_COLUMNS_V1
                time.sleep(pause)
                return raw
            if r.status_code == 404:
                return None
            print(f"  {date_str}: HTTP {r.status_code} (throttled?), retry {attempt+1}/{retries}")
            time.sleep(2 ** attempt + 1)
        except Exception as e:
            print(f"  {date_str}: {type(e).__name__}, retry {attempt+1}/{retries}")
            time.sleep(2 ** attempt + 1)
    print(f"  {date_str}: gave up after {retries} tries")
    return None

In [8]:
def filter_events(df, countries=COUNTRIES, event_root_codes=EVENT_ROOT_CODES,
                   min_mentions=MIN_MENTIONS, min_sources=MIN_SOURCES):
    df = df.copy()
    for c in ["NumMentions","NumSources","NumArticles","GoldsteinScale","AvgTone"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if countries is not None:
        df = df[
            df["Actor1CountryCode"].isin(countries) |
            df["Actor2CountryCode"].isin(countries) |
            df["ActionGeo_CountryCode"].isin(countries)
        ]
    if df.empty:
        return pd.DataFrame()

    if event_root_codes is not None:
        df = df[df["EventRootCode"].isin(event_root_codes)]
    if df.empty:
        return pd.DataFrame()

    df = df[(df["NumMentions"] >= min_mentions) & (df["NumSources"] >= min_sources)]
    if df.empty:
        return pd.DataFrame()

    return df[[c for c in KEEP_COLS if c in df.columns]]

In [9]:
def extract_events_for_range(start_date, end_date, output_path):
    all_events = []
    for d in tqdm(generate_gdelt_dates(start_date, end_date)):
        df = download_gdelt_file_v1(d)
        if df is None or df.empty:
            continue
        filt = filter_events(df)
        if not filt.empty:
            all_events.append(filt)
    if not all_events:
        print("No matching events found."); return pd.DataFrame()
    result = pd.concat(all_events, ignore_index=True)
    result["date"] = pd.to_datetime(result["SQLDATE"], format="%Y%m%d", errors="coerce")
    result = result.drop_duplicates(subset=["GLOBALEVENTID"])
    result.to_parquet(output_path, index=False)
    print(f"Saved {len(result):,} events to {output_path}")
    return result

def pull_gdelt_years(y0=START_YEAR, y1=END_YEAR):
    """Checkpointed, resumable pull -- safe to stop and re-run."""
    chunk_files = []
    for start, end in month_ranges(y0, y1):
        tag = start[:7]
        out = f"{CHECKPOINT_DIR}/gdelt_{tag}.parquet"
        chunk_files.append(out)
        if os.path.exists(out):
            print(f"✓ {tag} done — skipping"); continue
        print(f"\n=== {tag} ===")
        extract_events_for_range(start, end, output_path=out)
    parts = [pd.read_parquet(f) for f in chunk_files if os.path.exists(f)]
    events = pd.concat(parts, ignore_index=True).drop_duplicates("GLOBALEVENTID")
    events.to_parquet(OUTPUT_PATH, index=False)
    print(f"\nFINAL: {len(events):,} events -> {OUTPUT_PATH}")
    return events

## Test on one week before committing to the full pull

Sanity-checks the schema and gives you a row-count-per-day sense of scale before you commit to
a multi-hour, multi-year run.

In [10]:
test = extract_events_for_range("2022-01-01", "2022-01-07", output_path="test_events.parquet")
test.shape

100%|██████████| 7/7 [00:16<00:00,  2.35s/it]


Saved 111,414 events to test_events.parquet


(111414, 19)

## Full pull — uncomment once the test above looks right

In [ ]:
events = pull_gdelt_years(START_YEAR, END_YEAR)

✓ 2017-01 done — skipping
✓ 2017-02 done — skipping
✓ 2017-03 done — skipping
✓ 2017-04 done — skipping
✓ 2017-05 done — skipping
✓ 2017-06 done — skipping
✓ 2017-07 done — skipping
✓ 2017-08 done — skipping
✓ 2017-09 done — skipping
✓ 2017-10 done — skipping
✓ 2017-11 done — skipping
✓ 2017-12 done — skipping
✓ 2018-01 done — skipping
✓ 2018-02 done — skipping
✓ 2018-03 done — skipping
✓ 2018-04 done — skipping
✓ 2018-05 done — skipping
✓ 2018-06 done — skipping
✓ 2018-07 done — skipping
✓ 2018-08 done — skipping
✓ 2018-09 done — skipping
✓ 2018-10 done — skipping
✓ 2018-11 done — skipping
✓ 2018-12 done — skipping
✓ 2019-01 done — skipping
✓ 2019-02 done — skipping
✓ 2019-03 done — skipping
✓ 2019-04 done — skipping
✓ 2019-05 done — skipping
✓ 2019-06 done — skipping
✓ 2019-07 done — skipping
✓ 2019-08 done — skipping
✓ 2019-09 done — skipping
✓ 2019-10 done — skipping
✓ 2019-11 done — skipping
✓ 2019-12 done — skipping
✓ 2020-01 done — skipping
✓ 2020-02 done — skipping
✓ 2020-03 do

In [1]:
import glob, pyarrow as pa, pyarrow.parquet as pq
import pandas as pd

chunk_files = sorted(glob.glob("gdelt_chunks_full/gdelt_*.parquet"))
print(f"found {len(chunk_files)} chunk files")

writer, total, seen = None, 0, set()
for f in chunk_files:
    ch = pd.read_parquet(f)
    ch = ch.drop_duplicates("GLOBALEVENTID")          # within-chunk dedup
    total += len(ch)
    table = pa.Table.from_pandas(ch, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter("gdelt_events_2017_2023.parquet", table.schema)
    writer.write_table(table)
    print(f"  {f}: +{len(ch):,}  (total {total:,})")
if writer:
    writer.close()
print(f"DONE -> gdelt_events_2017_2023.parquet  ({total:,} rows)")

found 84 chunk files
  gdelt_chunks_full/gdelt_2017-01.parquet: +1,160,913  (total 1,160,913)
  gdelt_chunks_full/gdelt_2017-02.parquet: +1,145,411  (total 2,306,324)
  gdelt_chunks_full/gdelt_2017-03.parquet: +1,201,730  (total 3,508,054)
  gdelt_chunks_full/gdelt_2017-04.parquet: +1,050,730  (total 4,558,784)
  gdelt_chunks_full/gdelt_2017-05.parquet: +1,108,977  (total 5,667,761)
  gdelt_chunks_full/gdelt_2017-06.parquet: +1,013,787  (total 6,681,548)
  gdelt_chunks_full/gdelt_2017-07.parquet: +942,000  (total 7,623,548)
  gdelt_chunks_full/gdelt_2017-08.parquet: +997,936  (total 8,621,484)
  gdelt_chunks_full/gdelt_2017-09.parquet: +968,021  (total 9,589,505)
  gdelt_chunks_full/gdelt_2017-10.parquet: +993,746  (total 10,583,251)
  gdelt_chunks_full/gdelt_2017-11.parquet: +1,011,191  (total 11,594,442)
  gdelt_chunks_full/gdelt_2017-12.parquet: +868,523  (total 12,462,965)
  gdelt_chunks_full/gdelt_2018-01.parquet: +954,872  (total 13,417,837)
  gdelt_chunks_full/gdelt_2018-02.parq

In [ ]:
import pandas as pd
ev = pd.read_parquet("gdelt_events_2017_2023.parquet")
print("rows:", len(ev))
print("years:", sorted(pd.to_datetime(ev["date"]).dt.year.unique()))
print("distinct actor countries:", ev["Actor1CountryCode"].nunique())   # should be MANY more than the ~14 chip countries

In [1]:
import pyarrow.parquet as pq
pf = pq.ParquetFile("gdelt_events_2017_2023.parquet")
print("rows:", pf.metadata.num_rows)                     # reads metadata only, no data loaded
print("columns:", pf.schema.names)

# sample just the first row group for a peek (small, safe)
sample = pf.read_row_group(0).to_pandas()
print("distinct actor countries in sample:", sample["Actor1CountryCode"].nunique())
print(sample[["date","Actor1CountryCode","Actor2CountryCode","EventRootCode"]].head())

rows: 67487378
columns: ['GLOBALEVENTID', 'SQLDATE', 'DATEADDED', 'Actor1Name', 'Actor1CountryCode', 'Actor2Name', 'Actor2CountryCode', 'ActionGeo_CountryCode', 'ActionGeo_FullName', 'EventCode', 'EventBaseCode', 'EventRootCode', 'GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone', 'SOURCEURL', 'date']
distinct actor countries in sample: 215
        date Actor1CountryCode Actor2CountryCode EventRootCode
0 2016-01-02              None              None            04
1 2016-01-02              None              None            01
2 2016-01-02              None              None            04
3 2016-01-02               NZL              None            02
4 2016-01-02               USA              None            19


**After this runs**, point the downstream benchmark notebooks at `gdelt_events_2017_2023.parquet`
instead of `gdelt_chip_events_2017_2023.parquet`. Nothing else needs to change — the `CRIT`
filter, the ISO3→M49 mapping, and `add_gdelt`/`edge_features` are all unaffected; they'll just
see more rows to select from than before.

In [1]:
import pyarrow.parquet as pq
pf = pq.ParquetFile("gdelt_events_2017_2023.parquet")
print("rows:", pf.metadata.num_rows)
print("row groups:", pf.num_row_groups)      # tells us the natural chunk count for streaming
print("columns:", pf.schema.names)

rows: 67487378
row groups: 93
columns: ['GLOBALEVENTID', 'SQLDATE', 'DATEADDED', 'Actor1Name', 'Actor1CountryCode', 'Actor2Name', 'Actor2CountryCode', 'ActionGeo_CountryCode', 'ActionGeo_FullName', 'EventCode', 'EventBaseCode', 'EventRootCode', 'GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone', 'SOURCEURL', 'date']


In [2]:
import pyarrow.parquet as pq
import numpy as np, pandas as pd
from country_codes import ISO3_2_M49

COLS = ["GLOBALEVENTID","Actor1CountryCode","Actor2CountryCode",
        "EventRootCode","GoldsteinScale","NumMentions","AvgTone","date"]
CRIT = ["14","17","18","19","20"]
K = 10
iso2m49 = {**ISO3_2_M49, "TWN": "490"}

def stream_score(path="gdelt_events_2017_2023.parquet"):
    pf = pq.ParquetFile(path)
    # accumulators: dict keyed by (country/pair, ym) -> partial stats
    node_acc, pair_acc = {}, {}
    node_topk, pair_topk = {}, {}   # for top-k: keep a bounded list of top events per group

    def add(acc, key, score, gold, ment, tone):
        s = acc.get(key)
        if s is None:
            acc[key] = [1, score, score, gold*max(ment,1), max(ment,1), tone]  # n, ssum, smax, gm, msum, tsum
        else:
            s[0]+=1; s[1]+=score; s[2]=max(s[2],score); s[3]+=gold*max(ment,1); s[4]+=max(ment,1); s[5]+=tone

    def add_topk(acc, key, score, gold, ment, tone):
        buf = acc.setdefault(key, [])
        buf.append((score, gold, ment, tone))
        if len(buf) > 200:                       # bound the buffer; keep best 200, trim later to K
            buf.sort(key=lambda x:-x[0]); del buf[100:]

    for b in range(pf.num_row_groups):
        ch = pf.read_row_group(b, columns=COLS).to_pandas()
        ch = ch.dropna(subset=["Actor1CountryCode","date"])
        ch["EventRootCode"] = ch["EventRootCode"].astype(str)
        ch = ch[ch["EventRootCode"].isin(CRIT)]           # EARLY filter — drops most rows
        if ch.empty: continue
        ch["ym"]   = pd.to_datetime(ch["date"],errors="coerce").dt.to_period("M")
        ch = ch.dropna(subset=["ym"])
        g = pd.to_numeric(ch["GoldsteinScale"],errors="coerce").fillna(0).values
        m = pd.to_numeric(ch["NumMentions"],errors="coerce").fillna(0).values
        t = pd.to_numeric(ch["AvgTone"],errors="coerce").fillna(0).values
        sc = np.abs(g)*m
        a1 = ch["Actor1CountryCode"].values; a2 = ch["Actor2CountryCode"].values
        ym = ch["ym"].values
        for i in range(len(ch)):
            add(node_acc, (a1[i],ym[i]), sc[i],g[i],m[i],t[i])
            add_topk(node_topk, (a1[i],ym[i]), sc[i],g[i],m[i],t[i])
            if pd.notna(a2[i]):
                add(pair_acc, (a1[i],a2[i],ym[i]), sc[i],g[i],m[i],t[i])
                add_topk(pair_topk,(a1[i],a2[i],ym[i]), sc[i],g[i],m[i],t[i])
        if b % 10 == 0: print(f"  row-group {b}/{pf.num_row_groups}")
    return node_acc, pair_acc, node_topk, pair_topk

node_acc, pair_acc, node_topk, pair_topk = stream_score()
print("done streaming.")

  row-group 0/93
  row-group 10/93
  row-group 20/93
  row-group 30/93
  row-group 40/93
  row-group 50/93
  row-group 60/93
  row-group 70/93
  row-group 80/93
  row-group 90/93
done streaming.


In [3]:
def _finalize_node(acc, topk, out, use_topk):
    rows=[]
    src = topk if use_topk else acc
    for key,val in (topk.items() if use_topk else acc.items()):
        iso, ym = key
        if use_topk:
            buf = sorted(val, key=lambda x:-x[0])[:K]
            n=len(buf); ssum=sum(x[0] for x in buf); smax=max(x[0] for x in buf)
            gm=sum(x[1]*max(x[2],1) for x in buf); msum=sum(max(x[2],1) for x in buf); tsum=sum(x[3] for x in buf)
        else:
            n,ssum,smax,gm,msum,tsum = val
        rows.append((iso, ym.year, n, ssum/n, smax, gm/max(msum,1), tsum/n))
    df = pd.DataFrame(rows, columns=["iso3","year","n_events","score_mean","score_max","goldstein_wmean","tone_mean"])
    # roll month->year: aggregate the monthly rows to annual
    ann = df.groupby(["iso3","year"]).agg(
        events_total=("n_events","sum"), score_mean=("score_mean","mean"),
        score_max=("score_max","max"), score_vol=("score_mean","std"),
        goldstein_wmean=("goldstein_wmean","mean"), tone_mean=("tone_mean","mean"),
        active_months=("n_events", lambda s:(s>0).sum())).reset_index()
    ann["reporterCode"]=ann["iso3"].map(iso2m49); ann=ann.dropna(subset=["reporterCode"])
    ann["reporterCode"]=ann["reporterCode"].astype(int)
    ann.to_parquet(out,index=False); print("saved",out,ann.shape)

_finalize_node(node_acc, node_topk, "gdelt_features_by_country_year.parquet", use_topk=False)
_finalize_node(node_acc, node_topk, "gdelt_features_topk.parquet",            use_topk=True)

saved gdelt_features_by_country_year.parquet (1833, 10)
saved gdelt_features_topk.parquet (1833, 10)


In [4]:
cy = pd.read_parquet("gdelt_features_by_country_year.parquet")
print("shape:", cy.shape)
print("distinct countries (iso3):", cy["iso3"].nunique())
print("distinct reporterCodes:", cy["reporterCode"].nunique())
print("years:", sorted(cy["year"].unique()))
print(cy["iso3"].unique()[:40])

shape: (1833, 10)
distinct countries (iso3): 199
distinct reporterCodes: 199
years: [1920, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
['ABW' 'AFG' 'AGO' 'AIA' 'ALB' 'AND' 'ARE' 'ARG' 'ARM' 'ATG' 'AUS' 'AUT'
 'AZE' 'BDI' 'BEL' 'BEN' 'BFA' 'BGD' 'BGR' 'BHR' 'BHS' 'BLR' 'BLZ' 'BMU'
 'BOL' 'BRA' 'BRB' 'BRN' 'BTN' 'BWA' 'CAF' 'CAN' 'CHE' 'CHL' 'CHN' 'CIV'
 'CMR' 'COD' 'COG' 'COK']


In [5]:
for f in ["gdelt_features_by_country_year.parquet", "gdelt_features_topk.parquet"]:
    cy = pd.read_parquet(f)
    before = len(cy)
    cy = cy[cy["year"].between(2017, 2023)]
    cy.to_parquet(f, index=False)
    print(f"{f}: {before} -> {len(cy)} rows, years now {sorted(cy['year'].unique())}")

gdelt_features_by_country_year.parquet: 1833 -> 1377 rows, years now [2017, 2018, 2019, 2020, 2021, 2022, 2023]
gdelt_features_topk.parquet: 1833 -> 1377 rows, years now [2017, 2018, 2019, 2020, 2021, 2022, 2023]


In [6]:
def _finalize_pair(acc, topk, out, use_topk):
    rows=[]
    items = topk.items() if use_topk else acc.items()
    for key, val in items:
        a1, a2, ym = key
        if ym.year < 2017 or ym.year > 2023:        # year filter baked in
            continue
        if use_topk:
            buf = sorted(val, key=lambda x:-x[0])[:K]
            if not buf: continue
            n=len(buf); ssum=sum(x[0] for x in buf); smax=max(x[0] for x in buf)
            gm=sum(x[1]*max(x[2],1) for x in buf); msum=sum(max(x[2],1) for x in buf)
        else:
            n,ssum,smax,gm,msum,tsum = val
        rows.append((a1, a2, ym.year, n, ssum/n, smax, gm/max(msum,1)))
    df = pd.DataFrame(rows, columns=["Actor1CountryCode","Actor2CountryCode","year",
                                     "pair_events","pair_score_mean","pair_score_max","pair_gold_mean"])
    # roll pair-month -> pair-year
    ann = df.groupby(["Actor1CountryCode","Actor2CountryCode","year"]).agg(
        pair_events=("pair_events","sum"),
        pair_score_mean=("pair_score_mean","mean"),
        pair_score_max=("pair_score_max","max"),
        pair_gold_mean=("pair_gold_mean","mean")).reset_index()
    ann["reporterCode"]=ann["Actor1CountryCode"].map(iso2m49)
    ann["partnerCode"] =ann["Actor2CountryCode"].map(iso2m49)
    ann=ann.dropna(subset=["reporterCode","partnerCode"])
    ann["reporterCode"]=ann["reporterCode"].astype(int)
    ann["partnerCode"] =ann["partnerCode"].astype(int)
    ann.to_parquet(out, index=False); print("saved", out, ann.shape)

_finalize_pair(pair_acc, pair_topk, "gdelt_bilateral_by_pair_year.parquet", use_topk=False)
_finalize_pair(pair_acc, pair_topk, "gdelt_bilateral_topk.parquet",         use_topk=True)

saved gdelt_bilateral_by_pair_year.parquet (39048, 9)
saved gdelt_bilateral_topk.parquet (39048, 9)


In [7]:
for f in ["gdelt_bilateral_by_pair_year.parquet","gdelt_bilateral_topk.parquet"]:
    b = pd.read_parquet(f)
    print(f"\n{f}: {b.shape}")
    print("  years:", sorted(b['year'].unique()))
    print("  distinct pairs:", b.groupby(['reporterCode','partnerCode']).ngroups)
    print("  cols:", list(b.columns))



gdelt_bilateral_by_pair_year.parquet: (39048, 9)
  years: [2017, 2018, 2019, 2020, 2021, 2022, 2023]
  distinct pairs: 12209
  cols: ['Actor1CountryCode', 'Actor2CountryCode', 'year', 'pair_events', 'pair_score_mean', 'pair_score_max', 'pair_gold_mean', 'reporterCode', 'partnerCode']

gdelt_bilateral_topk.parquet: (39048, 9)
  years: [2017, 2018, 2019, 2020, 2021, 2022, 2023]
  distinct pairs: 12209
  cols: ['Actor1CountryCode', 'Actor2CountryCode', 'year', 'pair_events', 'pair_score_mean', 'pair_score_max', 'pair_gold_mean', 'reporterCode', 'partnerCode']


In [8]:
import pyarrow.parquet as pq
pf = pq.ParquetFile("all_products.parquet")
print("rows:", pf.metadata.num_rows, "| row groups:", pf.num_row_groups)
print("columns:", pf.schema.names)
s = pf.read_row_group(0).to_pandas()
print("\ngravity non-null in sample:")
for c in ["gdpcap_o","gdpcap_d","pop_o","pop_d","dist"]:
    if c in s: print(f"  {c}: {s[c].notna().mean()*100:.1f}% present")
if "isAggregate" in s: print("\nisAggregate values:", s["isAggregate"].value_counts(dropna=False).to_dict())

rows: 55401278 | row groups: 278
columns: ['refYear', 'reporterCode', 'flowCode', 'partnerCode', 'cmdCode', 'netWgt', 'FOBValue', 'primaryValue', 'isAggregate', 'pop_o', 'gdp_o', 'gdpcap_o', 'pop_d', 'gdp_d', 'gdpcap_d', 'dist', 'chapter', 'heading']

gravity non-null in sample:
  gdpcap_o: 100.0% present
  gdpcap_d: 98.1% present
  pop_o: 100.0% present
  pop_d: 98.7% present
  dist: 100.0% present

isAggregate values: {'1': 169838, '0': 30162}


In [9]:
ka = pd.read_parquet("gdelt_bilateral_by_pair_year.parquet")
tk = pd.read_parquet("gdelt_bilateral_topk.parquet")
merged = ka.merge(tk, on=['reporterCode','partnerCode','year'], suffixes=('_ka','_tk'))
print("pair-years where events differ:", (merged['pair_events_ka'] != merged['pair_events_tk']).sum(), '/', len(merged))

pair-years where events differ: 6533 / 39048


In [10]:
import pyarrow as pa, pyarrow.parquet as pq
from country_codes import DROP_CODES   # bloc codes (ASEAN, EU, ...), keeps 490=Taiwan

def filter_comtrade_stream(src="all_products.parquet", out="all_products_ready.parquet"):
    pf = pq.ParquetFile(src)
    writer, total, kept = None, 0, 0
    for b in range(pf.num_row_groups):
        ch = pf.read_row_group(b).to_pandas()
        total += len(ch)
        # 1) drop aggregate rows
        ch = ch[ch["isAggregate"].astype(str) == "0"]
        # 2) drop bloc/aggregate country codes (keep 490 = Taiwan)
        ch = ch[~ch["reporterCode"].astype(str).isin(DROP_CODES)]
        ch = ch[~ch["partnerCode"].astype(str).isin(DROP_CODES)]
        # 3) keep only rows with a usable target
        ch["primaryValue"] = pd.to_numeric(ch["primaryValue"], errors="coerce")
        ch = ch[ch["primaryValue"] > 0]
        if ch.empty: continue
        kept += len(ch)
        table = pa.Table.from_pandas(ch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out, table.schema)
        writer.write_table(table)
        if b % 25 == 0: print(f"  row-group {b}/{pf.num_row_groups} | kept {kept:,}/{total:,}")
    if writer: writer.close()
    print(f"DONE -> {out}  (kept {kept:,} of {total:,} = {100*kept/total:.1f}%)")

filter_comtrade_stream()

  row-group 0/278 | kept 30,162/200,000
  row-group 25/278 | kept 2,771,654/5,200,000
  row-group 50/278 | kept 4,278,561/10,200,000
  row-group 75/278 | kept 7,164,922/15,200,000
  row-group 100/278 | kept 8,882,910/20,200,000
  row-group 125/278 | kept 11,439,575/25,200,000
  row-group 150/278 | kept 13,512,877/30,200,000
  row-group 175/278 | kept 17,145,088/35,200,000
  row-group 200/278 | kept 18,998,856/40,200,000
  row-group 225/278 | kept 21,535,332/45,200,000
  row-group 250/278 | kept 24,135,587/50,200,000
  row-group 275/278 | kept 26,195,731/55,200,000
DONE -> all_products_ready.parquet  (kept 26,316,572 of 55,401,278 = 47.5%)
